In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [ ]:
def quantum_rng(num_bits):
    """Generates a list of random bits using a quantum circuit."""
    from qiskit import QuantumCircuit
    from qiskit.providers.basic_provider import BasicSimulator
    from qiskit import transpile
    
    rng_circuits = []
    for _ in range(num_bits):
        qc = QuantumCircuit(1, 1)
        # Apply Hadamard to state |0> to create equal superposition
        qc.h(0)
        # Measure the qubit to collapse the state to 0 or 1 randomly
        qc.measure(0, 0)
        rng_circuits.append(qc)
        
    simulator = BasicSimulator()
    transpiled_circuits = transpile(rng_circuits, simulator)
    job = simulator.run(transpiled_circuits, shots=1)
    result = job.result()
    
    bits = []
    for i in range(num_bits):
        counts = result.get_counts(i)
        bit = list(counts.keys())[0]
        bits.append(int(bit))
    return bits

N = 50 # Number of qubits for the protocol
print(f"Quantum RNG test (5 bits): {quantum_rng(5)}")

In [ ]:
# ==========================================
# ALICE'S PART
# ==========================================

print("--- ALICE ---")
# 1. Alice generates random bits for her key and bases
alice_bits = quantum_rng(N)
alice_bases = quantum_rng(N)

print(f"Alice's bits:  {alice_bits[:10]}... (Total: {len(alice_bits)})")
print(f"Alice's bases: {alice_bases[:10]}... (0=Z, 1=X)")

# 2. Alice encodes the key bits into quantum states
qubits = []
for i in range(N):
    qc = QuantumCircuit(1, 1)
    # Encode bit value (0 -> |0>, 1 -> |1>)
    if alice_bits[i] == 1:
        qc.x(0)
    # Encode in chosen basis (if X basis, apply Hadamard)
    if alice_bases[i] == 1:
        qc.h(0)
    qubits.append(qc)

print(f"Alice prepared {len(qubits)} qubits.")

In [ ]:
# ==========================================
# BOB'S PART
# ==========================================

print("\n--- BOB ---")
# 3. Bob generates random bits for his measurement bases
bob_bases = quantum_rng(N)
print(f"Bob's bases:   {bob_bases[:10]}...")

# 4. Bob receives the qubits and measures them in his chosen bases
bob_results = []
simulator = BasicSimulator()

for i in range(N):
    qc = qubits[i]
    # If Bob's basis is 1 (X basis), he needs to apply H before measurement
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)
    
    # Run the circuit
    transpiled_qc = transpile(qc, simulator)
    job = simulator.run(transpiled_qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    bit = int(list(counts.keys())[0])
    bob_results.append(bit)

print(f"Bob's results: {bob_results[:10]}...")

In [ ]:
# ==========================================
# PUBLIC DISCUSSION & SIFTING
# ==========================================

print("\n--- PUBLIC DISCUSSION ---")
# 5. Alice and Bob publicly share their bases and keep the bits where they matched
sifted_alice = []
sifted_bob = []
for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print(f"Sifted Key length: {len(sifted_alice)}")
print(f"Sifted Key (Alice): {sifted_alice[:10]}...")
print(f"Sifted Key (Bob):   {sifted_bob[:10]}...")

In [ ]:
# ==========================================
# ERROR CHECKING
# ==========================================

print("\n--- ERROR CHECKING ---")
# 6. Alice and Bob compare a random subset to check for interference
# We'll just use the first half of the sifted key for checking
num_check = len(sifted_alice) // 2
check_alice = sifted_alice[:num_check]
check_bob = sifted_bob[:num_check]

errors = 0
for i in range(num_check):
    if check_alice[i] != check_bob[i]:
        errors += 1
        
error_rate = errors / num_check if num_check > 0 else 0
print(f"Bits checked: {num_check}")
print(f"Errors found: {errors}")
print(f"Error Rate: {error_rate*100:.2f}%")

# Threshold typically around 10-11% for BB84
if error_rate > 0.1:
    print("\n🚨 ATTACK DETECTED! Aborting protocol.")
else:
    print("\n✅ No attack detected. Proceeding to use the final key.")
    final_key = sifted_alice[num_check:]
    print(f"Final Secret Key: {final_key}")